# NLP HW02 decode profile

thisarivson, private, Internet off, одна T4/GPU0. Один запуск, без обучения и голосования.
Входы: thisarivson/nlp-hw02-l3-live и thisarivson/nlp-hw02-26b-inputs. Только C-main L3 l2-62280, SHA проверяется.

Сначала full test1919 baseline/all × greedy/K4 × без нагрузки/с нагрузкой исполнения; greedy token IDs сравниваются после дополнения PAD до512, то есть ответы побитово. После — парные абляции отдельных флагов на192 вопросах, равномерно по рангу длины test1919. Их нельзя выдавать за full1919 throughput. SCOPE='full' переключает все абляции на1919, но матрица может не уложиться в лимит. Таблица всегда содержит scope/questions.

Флаги: share_cross; async_eos (32 шага вместо синхронных16); compact_done; race_sampler; lean_greedy; workers2; nice10; all. Предвыделенный self-KV, однократный cross-KV и Torch threads1 уже есть в baseline: отмечены как тождественные, без фиктивных повторов. CUDA graphs отклонены из-за динамического prefix/адресов/форм и риска изменения greedy. Для compaction GPU bit-exact — обязательный gate, не предположение. RNG-поток сэмплов может измениться; softmax/temperature и категориальный закон сохранены.

Нагрузка — непрерывные синтетические численные программы через реальный frozen executor: базово4 workers/nice0, отдельно2 workers или nice10, вместе2/nice10. Это контролируемая нагрузка, не воспроизведение реального распределения кандидатов. После генерации драйвер завершает сервис; рабочая final inference пока не меняется.

Working deadline650с, watchdog690с, внешний предел720с от первой code cell. summary.json/timings.csv сохраняются после каждого замера. Четыре32-step torch.profiler trace (.json.gz), K4/1024 lanes, и summary: kernels/CPU ops/launch/sync, доли перекрываются; GPU idle — верхняя оценка host overhead, а не точное Python time. budget_stop_partial не проходит GPU gate. Цель2× выводится из full baseline/all; успешность и качество K8+K8 требуют следующего отдельного end-to-end прогона.

In [ ]:
import json, os, signal, subprocess, sys, threading, time
from pathlib import Path
SESSION_START = time.monotonic()
OUTPUT = Path('/kaggle/working/decode-profile')
MODULES = OUTPUT/'modules'
MODULES.mkdir(parents=True, exist_ok=True)
SOURCES = {'final_core.py': "# Generated by build_final_inference.py from exp26_core.py; do not edit.\nimport math,time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\nVOCAB=8192\nMAX_POSITIONS=2048\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError('Position overflow')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b''.join(chunks)\n\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n\nclass BudgetExpired(Exception):pass\n\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired('Session budget reached; completed run boundaries are preserved')\n\n\ndef sync(device):\n if device.type=='cuda':torch.cuda.synchronize(device)\n\n", 'final_fast.py': '# Generated by build_final_inference.py from exp27b_fast.py; do not edit.\nimport math\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\nclass Context:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads;self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,:];self.positions=model.positions(torch.arange(capacity,device=src.device));self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            a=layer.multihead_attn;k,v=F.linear(memory,a.in_proj_weight[self.width:],a.in_proj_bias[self.width:]).chunk(2,-1)\n            self.cross.append(tuple(t.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous() for t in (k,v)))\n\n\nclass Decoder:\n    @torch.inference_mode()\n    def __init__(self,context,k):\n        self.c,self.k,self.t=context,k,0;self.n=len(context.src)*k\n        self.allowed=context.allowed.repeat_interleave(k,dim=0)\n        self.cross=[tuple(t.repeat_interleave(k,dim=0).contiguous() for t in pair) for pair in context.cross]\n        self.cross_bytes=sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape=(self.n,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n        self.valid=torch.zeros(self.n,context.capacity,dtype=torch.bool,device=context.src.device)\n    def heads(self,x):return x.reshape(self.n,1,self.c.heads,self.c.dim).transpose(1,2)\n    def attend(self,a,q,k,v,mask):\n        assert q.ndim==k.ndim==v.ndim==4\n        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)\n        return a.out_proj(y.transpose(1,2).contiguous().reshape(self.n,1,self.c.width))\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t;x=c.model.tokens(tokens[:,None])*math.sqrt(c.width)+c.positions[t][None,None,:]\n        self.valid[:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,self.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,t:t+1]=self.heads(k);sv[:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:t+1],sv[:,:,:t+1],self.valid[:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,self.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,0])\n\n\nfrom final_decode import decode\n', 'final_decode.py': '"""V3 decode loop; frozen Context/Decoder mathematics live in final_fast."""\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, check_every=16, compute_logprob=True):\n    from final_fast import Decoder\n    d = Decoder(context, k)\n    device = context.src.device\n    token = torch.full((d.n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(d.n, dtype=torch.bool, device=device)\n    # KV, output, running LP/length buffers are allocated once, never concatenated.\n    output = torch.full((d.n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums = torch.zeros(d.n, device=device)\n    lengths = torch.zeros(d.n, dtype=torch.long, device=device)\n    for t in range(context.capacity):\n        if t % check_every == 0:\n            C.check_deadline(deadline)\n        logits = d.step(token)\n        if temperature is not None or compute_logprob:\n            logits = logits.float()\n        logits[:, :2] = -torch.inf  # PAD/BOS: contiguous slice avoids a per-step index tensor.\n        pick = logits.argmax(-1) if temperature is None else torch.multinomial(\n            F.softmax(logits / temperature, -1), 1).squeeze(-1)\n        if compute_logprob:\n            logprob = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n            sums.add_(torch.where(done, 0., logprob))\n        lengths.add_(~done)\n        token = pick.masked_fill(done, C.PAD)\n        output[:, t] = token\n        done |= token.eq(C.EOS)\n        if (t + 1) % check_every == 0 and bool(done.all().cpu()):\n            break\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (sums / lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_representation.py': "# Generated by build_final_inference.py from s3_representation.py; do not edit.\nfrom tokenizers import Tokenizer,AddedToken\nMODES={'header_tail':'<|header_tail|>','header':'<|header|>','raw':'<|raw|>'}\nNUMBERS=[f'<|N{i}|>' for i in range(1,21)]\n\ndef restore_compact(text):\n    mode = next((k for k, token in MODES.items() if text.startswith(token)), None)\n    if mode is None:\n        raise ValueError('Missing representation mode')\n    fragment = text[len(MODES[mode]):]\n    if mode == 'raw':\n        return fragment\n    if mode == 'header':\n        return 'def solve() -> int:\\n' + fragment\n    lines = fragment.splitlines(keepends=True)\n    if not lines or not lines[-1].startswith('    '):\n        raise ValueError('Missing final result expression')\n    lines[-1] = '    result = ' + lines[-1][4:]\n    body = ''.join(lines)\n    return 'def solve() -> int:\\n' + body + ('' if body.endswith('\\n') else '\\n') + '    return result\\n'\n\n\ndef tokenizer(path, variant):\n    tok = Tokenizer.from_file(str(path))\n    assert tok.get_vocab_size() == 8189\n    added = list(MODES.values()) if variant == 30 else NUMBERS\n    assert tok.add_special_tokens([AddedToken(t, normalized=False, special=True) for t in added]) == len(added)\n    # Original byte alphabet is unchanged; added specials decode literally, never get skipped.\n    visible = list(range(33,127)) + list(range(161,173)) + list(range(174,256))\n    bchars = {b:chr(b) for b in visible}\n    for i,b in enumerate(b for b in range(256) if b not in bchars):\n        bchars[b] = chr(256+i)\n    charbytes = {c:b for b,c in bchars.items()}\n    tokenbytes = [b'',b'',b''] + [bytes(charbytes[c] for c in tok.id_to_token(i)) for i in range(8189)]\n    tokenbytes += [t.encode() for t in added]\n    assert all(tok.encode(t, add_special_tokens=False).ids == [8189+i] for i,t in enumerate(added))\n    return tok, tokenbytes\n\n", 'final_executor.py': '"""Incremental global execution cache, usable while the GPU generates next batch."""\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass Executor:\n    def __init__(self, workers=None, service_path=None):\n        self.workers = workers if workers is not None else (os.cpu_count() or 1)\n        env = dict(os.environ, OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n                   PYTHONHASHSEED=\'0\')\n        self.child = subprocess.Popen([sys.executable, str(service_path or Path(__file__).with_name(\'final_execute.py\')),\n                                      str(self.workers)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                                      stderr=subprocess.DEVNULL, text=True, bufsize=1, env=env,\n                                      start_new_session=True)\n        self.codes = {}\n        self.results = {}\n        self.requests = 0\n        self.started = time.monotonic()\n        self.error = None\n        self.task_wall_seconds = self.task_cpu_seconds = 0.\n        self.first_submit = self.last_result = None\n        def read():\n            try:\n                for line in self.child.stdout:\n                    index, value, *metadata = json.loads(line)\n                    if metadata and metadata[0]:\n                        self.task_wall_seconds += metadata[0][\'task_wall_seconds\']\n                        self.task_cpu_seconds += metadata[0][\'task_cpu_seconds\']\n                    self.last_result = time.monotonic()\n                    self.results[index] = value\n            except Exception as error:\n                self.error = type(error).__name__\n        self.reader = threading.Thread(target=read, daemon=True)\n        self.reader.start()\n\n    def submit(self, candidates):\n        jobs = []\n        for candidate in candidates:\n            self.requests += 1\n            code = candidate[\'code\']\n            if code not in self.codes:\n                index = len(self.codes)\n                self.codes[code] = index\n                jobs.append([index, code])\n        if jobs:\n            if self.first_submit is None:\n                self.first_submit = time.monotonic()\n            self.child.stdin.write(json.dumps(jobs) + \'\\n\')\n            self.child.stdin.flush()\n\n    def attach(self, candidates):\n        for candidate in candidates:\n            index = self.codes[candidate[\'code\']]\n            candidate[\'execution\'] = self.results.get(index, {\'status\': \'budget_not_evaluated\'})\n\n    def drain(self, deadline):\n        while len(self.results) < len(self.codes) and time.monotonic() < deadline:\n            exit_code = self.child.poll()\n            if exit_code is not None:\n                self.error = self.error or \'executor_exit_\'+str(exit_code)\n                break\n            if self.error:\n                break\n            time.sleep(.002)\n        return len(self.results) == len(self.codes)\n\n    def summary(self):\n        return {\'workers\': self.workers, \'requested_candidates\': self.requests,\n                \'unique_programs\': len(self.codes), \'deduplicated\': self.requests-len(self.codes),\n                \'completed_unique\': len(self.results), \'service_wall_seconds\': time.monotonic()-self.started,\n                \'sum_worker_task_wall_seconds\': self.task_wall_seconds,\n                \'sum_worker_task_cpu_seconds\': self.task_cpu_seconds,\n                \'first_submit_to_last_result_seconds\': (self.last_result-self.first_submit\n                    if self.last_result is not None and self.first_submit is not None else None),\n                \'service_wall_includes_GPU_wait\': True,\n                \'wall_limit\': 3., \'cpu_limit\': 2., \'memory_bytes\': 768*1024**2,\n                \'overlap\': True, \'isolation\': \'fresh numeric namespaces / pristine SymPy fork\', \'error\': self.error}\n\n    def close(self):\n        # Never pool.join() unbounded in the CUDA process.\n        try:\n            os.killpg(self.child.pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        try:\n            self.child.wait(timeout=.1)\n        except subprocess.TimeoutExpired:\n            pass\n        self.child.stdin.close()\n        self.reader.join(timeout=.1)\n        self.child.stdout.close()\n', 'final_execute.py': '"""Persistent Torch-free service: fresh numeric namespaces, pristine SymPy forks.\n\nstdin: JSON lists of [id, code], or null to finish. stdout: [id, execution, metrics].\nThe launcher can kill the complete service process group, including its workers.\n"""\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport resource\nimport select\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\nimport final_numeric as numeric\n\nMEMORY = 768 * 1024**2\nWALL = 3.\nCPU = 2.\n\n\ndef kill_program(pid):\n    # Each program owns a group; kill descendants even after a successful result.\n    try:\n        os.killpg(pid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        try:\n            os.kill(pid, signal.SIGKILL)  # Child may not yet have called setsid().\n        except ProcessLookupError:\n            pass\n    until = time.monotonic()+.1\n    while time.monotonic() < until:\n        try:\n            if os.waitpid(pid, os.WNOHANG)[0]:\n                return\n        except ChildProcessError:\n            return\n        time.sleep(.001)\n    raise RuntimeError(\'Program could not be reaped; replace pristine worker\')\n\n\ndef isolated_task(code, tree, connection):\n    reader, writer = os.pipe()\n    parent = os.getpid()\n    pid = os.fork()  # This parent has no Torch/CUDA and no reader/BLAS threads.\n    if pid == 0:\n        try:\n            os.close(reader)\n            connection.close()\n            os.setsid()\n            if sys.platform.startswith(\'linux\'):\n                import ctypes\n                signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n                if ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) != 0 or os.getppid() != parent:\n                    os.killpg(os.getpgrp(), signal.SIGKILL)\n            resource.setrlimit(resource.RLIMIT_CPU, (int(CPU)+1, int(CPU)+1))\n            result = numeric.execute_task((code, WALL, CPU), tree=tree, isolated=True)\n            os.write(writer, json.dumps(result, separators=(\',\', \':\')).encode())\n        finally:\n            os._exit(0)\n    os.close(writer)\n    try:\n        # Same outer limit as v2; catches code that repeatedly catches timer exceptions.\n        if select.select([reader], [], [], WALL+2.)[0]:\n            data = os.read(reader, 4096)\n            if data:\n                return json.loads(data)\n        return {\'status\': \'timeout_or_worker_exit\'}\n    finally:\n        os.close(reader)\n        kill_program(pid)\n\n\ndef worker(connection):\n    # Library diagnostics and user output cannot corrupt the JSON protocol.\n    with open(os.devnull, \'w\') as sink:\n        os.dup2(sink.fileno(), 1)\n        os.dup2(sink.fileno(), 2)\n    numeric.worker_init(MEMORY)\n    pristine = None\n    while True:\n        item = connection.recv()\n        if item is None:\n            return\n        index, code = item\n        started = time.monotonic()\n        used = time.process_time()\n        before = resource.getrusage(resource.RUSAGE_CHILDREN)\n        tree, status = numeric.filter_program(code)\n        if status:\n            result = {\'status\': status}\n        elif any((isinstance(node, numeric.ast.Import) and any(a.name == \'sympy\' for a in node.names)) or\n                 (isinstance(node, numeric.ast.ImportFrom) and node.module == \'sympy\')\n                 for node in numeric.ast.walk(tree)):\n            if pristine is None:\n                numeric.prepare_pristine_worker()\n                pristine = dict(numeric._MODULES)\n            numeric._MODULES.clear()\n            numeric._MODULES.update(pristine)\n            result = isolated_task(code, tree, connection)\n        else:\n            # Only builtins and a fresh math namespace are reachable on this path.\n            # No user code can access SymPy objects/caches; env and timers are per task.\n            result = numeric.execute_task((code, WALL, CPU), tree=tree)\n        after = resource.getrusage(resource.RUSAGE_CHILDREN)\n        connection.send((index, result, {\'task_wall_seconds\': time.monotonic()-started,\n            \'task_cpu_seconds\': time.process_time()-used + after.ru_utime+after.ru_stime-before.ru_utime-before.ru_stime}))\n\n\ndef serve(workers):\n    inbox = queue.Queue()\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context(\'spawn\')\n    slots = []\n    pending = deque()\n    finished = False\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.]\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=.05)\n        pipe.close()\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n    try:\n        slots = [start() for _ in range(workers)]\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if pipe_failed or not process.is_alive() or (slot[2] is not None and time.monotonic()-submitted >= WALL+2.5):\n                    if slot[2] is not None:\n                        emit(slot[2], {\'status\': \'timeout_or_worker_exit\'})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(active_pipes, timeout=.002)  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == \'__main__\':\n    # Independent supervisor kills the process group on parent death (Linux).\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]))\n', 'final_numeric.py': '"""Final-only numeric policy; historical experiment executors remain unchanged."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport random\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\',\'random\',\'format\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Attribute) and isinstance(n.ctx,(ast.Store,ast.Del)):\n            return None,\'blocked_attribute_mutation\'\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    # All sympify bindings import this parser dynamically, including object methods.\n    # Symbol names use the separately validated symbols() syntax, never parse_expr.\n    from sympy.parsing import sympy_parser\n    def denied_parser(*args,**kwargs):\n        raise ValueError(\'String expression parsing forbidden\')\n    sympy_parser.parse_expr=denied_parser\n    from sympy.core.cache import clear_cache\n    clear_cache()\n    from sympy.core.random import seed\n    seed(0)\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef prepare_pristine_worker():\n    """Run once in a Torch-free, single-threaded parent; never run user code here."""\n    _MODULES.clear()\n    safe_import(\'math\')\n    safe_import(\'sympy\')  # Parser deny patch, cache clear and SymPy seed happen ONCE.\n    random.seed(0)\n\ndef execute_task(payload, tree=None, isolated=False):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    if not isolated:\n        _MODULES.clear()  # Historical/reference path; fork children inherit pristine modules.\n    random.seed(0)\n    if tree is None:\n        tree,status=filter_program(code)\n        if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.signal(signal.SIGPROF,lambda *_:(_ for _ in ()).throw(ProgramTimeout()))\n    signal.setitimer(signal.ITIMER_PROF,cpu_limit)\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:\n        signal.setitimer(signal.ITIMER_REAL,0)\n        signal.setitimer(signal.ITIMER_PROF,0)\n', 'final_speed.py': '"""Opt-in decoder experiments. Frozen final_fast/final_decode remain the control."""\nfrom dataclasses import dataclass\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\nimport final_fast as Fast\n\n\nGRAPH_REFUSAL = (\n    \'CUDA graphs disabled: prefix length, KV slice addresses and active batch change every step. \'\n    \'One graph per position/batch would require hundreds of captures; full-capacity masked SDPA \'\n    \'changes reduction shapes and cannot promise bit-exact greedy. No untested graph fallback.\')\n\n\n@dataclass(frozen=True)\nclass Options:\n    preallocated_kv: bool = True  # Already present in the frozen baseline; no new speedup.\n    share_cross: bool = False\n    async_eos: bool = False\n    eos_every: int = 16\n    compact_done: bool = False\n    compact_fraction: float = .25\n    race_sampler: bool = False\n    lean_greedy: bool = False\n    cuda_graphs: bool = False\n\n    def __post_init__(self):\n        if not self.preallocated_kv:\n            raise ValueError(\'Baseline already preallocates KV; a cat regression is not an optimization\')\n        if self.cuda_graphs:\n            raise NotImplementedError(GRAPH_REFUSAL)\n        if self.eos_every < 1 or not 0 < self.compact_fraction <= 1:\n            raise ValueError(\'Invalid EOS/compaction interval\')\n\n\nclass Decoder(Fast.Decoder):\n    @torch.inference_mode()\n    def __init__(self, context, k, share_cross=False):\n        if not share_cross or k != 1:\n            super().__init__(context, k)\n            return\n        # Immutable cross KV and source mask can be shared for greedy, no replication.\n        self.c, self.k, self.t, self.n = context, k, 0, len(context.src)\n        self.allowed, self.cross = context.allowed, context.cross\n        self.cross_bytes = sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape = (self.n, context.heads, context.capacity, context.dim)\n        self.self_kv = [(context.positions.new_empty(shape), context.positions.new_empty(shape))\n                        for _ in context.cross]\n        self.valid = torch.zeros(self.n, context.capacity, dtype=torch.bool, device=context.src.device)\n\n    @torch.inference_mode()\n    def select(self, index):\n        """Copy only initialized self-KV prefix, then continue at the SAME position."""\n        if index.ndim != 1 or not len(index):\n            raise ValueError(\'Nonempty 1-D active indices required\')\n        n, cap = len(index), self.c.capacity\n        self.allowed = self.allowed.index_select(0, index)\n        self.cross = [tuple(t.index_select(0, index) for t in pair) for pair in self.cross]\n        # Release each old layer as soon as its replacement is ready. Holding both\n        # complete caches would make compaction itself an avoidable T4 OOM peak.\n        for number in range(len(self.self_kv)):\n            pair = self.self_kv[number]\n            new_pair = []\n            for old in pair:\n                new = old.new_empty((n, self.c.heads, cap, self.c.dim))\n                new[:, :, :self.t].copy_(old[:, :, :self.t].index_select(0, index))\n                new_pair.append(new)\n            self.self_kv[number] = tuple(new_pair)\n        self.n = n\n        self.valid = self.valid.index_select(0, index)\n\n\ndef race_pick(probabilities, noise):\n    """Independent Exp(1) race: argmax(p/E) has categorical probabilities p.\n\n    Uses exactly the same FP32 softmax as baseline; RNG consumption can change.\n    Avoids multinomial\'s probability validation/reductions. Noise storage is reused.\n    """\n    noise.exponential_()\n    return torch.div(probabilities, noise, out=noise).argmax(-1)\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, compute_logprob=True,\n           options=None, stats=None):\n    options = options or Options()\n    d = Decoder(context, k, options.share_cross)\n    device, n = context.src.device, d.n\n    token = torch.full((n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(n, dtype=torch.bool, device=device)\n    output = torch.full((n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums, lengths = torch.zeros(n, device=device), torch.zeros(n, dtype=torch.long, device=device)\n    final_sums, final_lengths = torch.zeros_like(sums), torch.zeros_like(lengths)\n    original = torch.arange(n, device=device)\n    compacted = False\n    noise = None\n    host = event = None\n    pending = False\n    async_eos = options.async_eos and device.type == \'cuda\'\n    if async_eos:\n        host = torch.empty(n, dtype=torch.bool, pin_memory=True)\n        event = torch.cuda.Event()\n    history = []\n    checks = 0\n\n    def flush():\n        final_sums.index_copy_(0, original, sums)\n        final_lengths.index_copy_(0, original, lengths)\n\n    def inspect(snapshot):\n        nonlocal token, done, sums, lengths, original, compacted, noise, host\n        if bool(snapshot.all()):\n            return True\n        if not options.compact_done:\n            return False\n        keep = (~snapshot).nonzero().flatten()  # CPU only: never CUDA nonzero/item.\n        if options.compact_done and len(keep) <= d.n*(1-options.compact_fraction):\n            flush()\n            index = keep.to(device)\n            before = d.n\n            d.select(index)\n            token, done = token[index], done[index]\n            sums, lengths, original = sums[index], lengths[index], original[index]\n            compacted, noise = True, None\n            history.append({\'step\': d.t, \'before\': before, \'after\': d.n})\n            if async_eos:\n                host = torch.empty(d.n, dtype=torch.bool, pin_memory=True)\n        return False\n\n    try:\n        for t in range(context.capacity):\n            if t % 16 == 0:\n                C.check_deadline(deadline)\n            if pending and event.query():\n                pending = False\n                if inspect(host):\n                    break\n            logits = d.step(token)\n            if temperature is not None or compute_logprob:\n                logits = logits.float()\n            logits[:, :2] = -torch.inf\n            if temperature is None:\n                pick = logits.argmax(-1)\n            else:\n                probabilities = F.softmax(logits / temperature, -1)\n                if options.race_sampler:\n                    if noise is None:\n                        noise = torch.empty_like(probabilities)\n                    pick = race_pick(probabilities, noise)\n                else:\n                    pick = torch.multinomial(probabilities, 1).squeeze(-1)\n            if compute_logprob:\n                lp = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n                sums.add_(torch.where(done, 0., lp))\n            if compute_logprob or not options.lean_greedy:\n                lengths.add_(~done)\n            token = pick.masked_fill(done, C.PAD)\n            if compacted:\n                output[:, t].index_copy_(0, original, token)\n            else:\n                output[:, t] = token\n            done |= token.eq(C.EOS)\n            if (t+1) % options.eos_every == 0:\n                if async_eos:\n                    if not pending:\n                        host.copy_(done, non_blocking=True)\n                        event.record()\n                        pending = True\n                        checks += 1\n                else:\n                    checks += 1\n                    if inspect(done.cpu()):\n                        break\n    finally:\n        # Also retain pinned storage on deadline/OOM exceptions until copy finishes.\n        if pending:\n            event.synchronize()\n    flush()\n    if stats is not None:\n        stats.update(steps=d.t, eos_checks=checks, compactions=history,\n                     async_eos=async_eos, rng_stream_may_change=options.race_sampler or options.compact_done)\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (final_sums / final_lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_profile.py': '"""One bounded T4 session: paired decoder timings, full greedy gate, short traces."""\nimport argparse\nimport csv\nfrom dataclasses import asdict\nimport gc\nimport gzip\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport time\nimport zipfile\nimport torch\nfrom torch import nn\nimport final_core as C\nimport final_fast as Fast\nimport final_speed as Speed\nfrom final_representation import tokenizer\n\nINPUT_SHA = {\'tokenizer.json\': \'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\',\n             \'test.json\': \'20cacb334e3fc81291433d2173d9085b17a15e1834bd84aea67b350403182503\'}\nWEIGHT_SHA = \'8c11b1038855422b6912e243049a554317978f848edb412ed48aebbe4cb6b886\'\nVARIANTS = {\n    \'baseline\': {},\n    \'share_cross\': {\'share_cross\': True},\n    \'async_eos\': {\'async_eos\': True, \'eos_every\': 32},\n    \'compact_done\': {\'compact_done\': True},\n    \'race_sampler\': {\'race_sampler\': True},\n    \'lean_greedy\': {\'lean_greedy\': True},\n    \'workers2\': {},\n    \'nice10\': {},\n    \'all\': {\'share_cross\': True, \'async_eos\': True, \'eos_every\': 32,\n            \'compact_done\': True, \'race_sampler\': True, \'lean_greedy\': True}}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4*1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write(path, value):\n    temporary = Path(path).with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False)+\'\\n\')\n    temporary.replace(path)\n\n\ndef unique(root, name):\n    paths = list(Path(root).rglob(name))\n    assert len(paths) == 1, f\'Missing or ambiguous {name}: {paths}\'\n    return paths[0]\n\n\ndef prepare(root, target):\n    result = {}\n    for name, expected in INPUT_SHA.items():\n        direct = list(Path(root).rglob(name))\n        if direct:\n            assert len(direct) == 1\n            path = direct[0]\n        else:\n            archive = unique(root, \'26b-training-inputs-d0-initial-and-evaluation.zip\')\n            with zipfile.ZipFile(archive) as stream:\n                names = [p for p in stream.namelist() if Path(p).name == name]\n                assert len(names) == 1\n                path = target/name\n                path.write_bytes(stream.read(names[0]))\n        assert sha(path) == expected, name\n        result[name] = path\n    return result\n\n\ndef load_model(path, device):\n    assert sha(path) == WEIGHT_SHA, \'Require frozen L3 C-main l2-62280\'\n    payload = torch.load(path, weights_only=True, map_location=\'cpu\')\n    assert payload[\'format\'] == \'nlp-hw02-model-only-v1\' and not payload[\'optimizer_included\']\n    model = C.SimpleTransformer(**payload[\'model_spec\'], dropout=0.).eval().half()\n    model.load_state_dict(payload[\'model\'], strict=True)\n    return model.to(device)\n\n\nclass Load:\n    """Separate Torch-free driver; service owns and cleans its worker process group."""\n    def __init__(self, directory, workers, nice):\n        self.directory, self.workers, self.nice = Path(directory), workers, nice\n        self.child = None\n        self.info = {\'nice\': 0, \'nice_applied\': True}\n        self.log = None\n\n    def __enter__(self):\n        if not self.workers:\n            return self\n        self.directory.mkdir(parents=True, exist_ok=True)\n        for name in (\'ready.json\', \'load-summary.json\'):\n            (self.directory/name).unlink(missing_ok=True)\n        self.log = (self.directory/\'driver.log\').open(\'w\')\n        self.child = subprocess.Popen([sys.executable, str(Path(__file__).with_name(\'final_profile_load.py\')),\n            \'--workers\', str(self.workers), \'--nice\', str(self.nice), \'--directory\', str(self.directory)],\n            start_new_session=True, stdout=self.log, stderr=subprocess.STDOUT)\n        until = time.monotonic()+30\n        try:\n            while not (self.directory/\'ready.json\').exists():\n                if self.child.poll() is not None or time.monotonic() >= until:\n                    raise RuntimeError(\'Execution load startup failed\')\n                time.sleep(.02)\n            self.info = json.loads((self.directory/\'ready.json\').read_text())\n        except BaseException:\n            self.__exit__(*sys.exc_info())\n            raise\n        return self\n\n    def alive(self):\n        if self.child is not None and self.child.poll() is not None:\n            raise RuntimeError(\'Execution load stopped during timing\')\n\n    def __exit__(self, *_):\n        if self.child is not None and self.child.poll() is None:\n            self.child.terminate()\n            try:\n                self.child.wait(timeout=2.)\n            except subprocess.TimeoutExpired:\n                os.killpg(self.child.pid, signal.SIGKILL)\n                self.child.wait(timeout=2.)\n        if self.log is not None:\n            self.log.close()\n\n\ndef canonical(ids, cap):\n    result = torch.full((*ids.shape[:-1], cap), C.PAD, dtype=torch.long)\n    result[..., :ids.shape[-1]] = ids\n    return result\n\n\n@torch.inference_mode()\ndef measure(model, encoded, order, variant, k, device, deadline, cap=512, sequence_budget=1024):\n    """Same order, capacity and initial batch shapes for every paired comparison."""\n    options = Speed.Options(**VARIANTS[variant])\n    question_batch = sequence_budget//k\n    predictions, batches = {}, []\n    torch.manual_seed(270042)\n    torch.cuda.manual_seed_all(270042)\n    torch.cuda.synchronize(device)\n    torch.cuda.reset_peak_memory_stats(device)\n    gpu0, gpu1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n    gpu0.record()\n    begin, cpu = time.monotonic(), time.process_time()\n    for offset in range(0, len(order), question_batch):\n        C.check_deadline(deadline)\n        selected = order[offset:offset+question_batch]\n        b = time.monotonic()\n        src = nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in selected],\n                    batch_first=True, padding_value=C.PAD).to(device)\n        context = Fast.Context(model, src, cap)\n        stats = {}\n        if variant in (\'baseline\', \'workers2\', \'nice10\'):\n            ids, lps, _ = Fast.decode(context, k, None if k == 1 else .7,\n                        deadline=deadline, compute_logprob=k != 1)\n            stats[\'steps\'] = ids.shape[-1]\n        else:\n            ids, lps, _ = Speed.decode(context, k, None if k == 1 else .7,\n                        deadline=deadline, compute_logprob=k != 1, options=options, stats=stats)\n        if k == 1:\n            predictions.update(zip(selected, canonical(ids, cap)[:, 0]))\n        batches.append({\'questions\': len(selected), \'sequences\': len(selected)*k,\n                        \'source_tokens\': src.shape[1], \'seconds\': time.monotonic()-b, **stats})\n        del src, context, ids, lps\n    gpu1.record()\n    torch.cuda.synchronize(device)\n    seconds = time.monotonic()-begin\n    row = {\'variant\': variant, \'mode\': \'greedy\' if k == 1 else \'K4\', \'questions\': len(order),\n           \'sequences\': len(order)*k, \'seconds\': seconds, \'sequences_per_second\': len(order)*k/seconds,\n           \'parent_cpu_seconds\': time.process_time()-cpu, \'gpu_stream_span_seconds\': gpu0.elapsed_time(gpu1)/1000,\n           \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(device),\n           \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(device), \'cap\': cap,\n           \'sequence_budget\': sequence_budget, \'options\': asdict(options), \'batches\': batches}\n    # Event span includes GPU idle between host launches; it is NOT summed kernel time.\n    if predictions:\n        values = torch.stack([predictions[i] for i in order])\n        row[\'greedy_token_sha256\'] = hashlib.sha256(values.numpy().tobytes()).hexdigest()\n    return row, predictions\n\n\ndef union_us(intervals):\n    total, right = 0., float(\'-inf\')\n    for lo, hi in sorted(intervals):\n        total += max(0., hi-max(lo, right))\n        right = max(right, hi)\n    return total\n\n\ndef trace_summary(trace):\n    events = trace[\'traceEvents\']\n    window = next(e for e in events if e.get(\'name\') == \'decode_window\' and e.get(\'ph\') == \'X\')\n    lo, hi = window[\'ts\'], window[\'ts\']+window[\'dur\']\n    def intervals(predicate):\n        return [(max(lo, e[\'ts\']), min(hi, e[\'ts\']+e[\'dur\'])) for e in events\n            if e.get(\'ph\') == \'X\' and \'dur\' in e and e[\'ts\'] < hi and e[\'ts\']+e[\'dur\'] > lo and predicate(e)]\n    kernels = intervals(lambda e: e.get(\'cat\') == \'kernel\')\n    ops = intervals(lambda e: e.get(\'cat\') == \'cpu_op\' and e.get(\'tid\') == window[\'tid\'])\n    syncs = intervals(lambda e: e.get(\'cat\') == \'cuda_runtime\' and\n                     (\'Synchronize\' in e.get(\'name\', \'\') or \'Memcpy\' in e.get(\'name\', \'\')))\n    launches = intervals(lambda e: e.get(\'cat\') == \'cuda_runtime\' and \'Launch\' in e.get(\'name\', \'\'))\n    span = hi-lo\n    return {\'window_us\': span, \'kernel_count\': len(kernels), \'kernel_busy_fraction\': union_us(kernels)/span,\n            \'gpu_idle_fraction_upper_bound_for_host_overhead\': 1-union_us(kernels)/span,\n            \'cpu_operator_fraction\': union_us(ops)/span, \'sync_and_memcpy_api_fraction\': union_us(syncs)/span,\n            \'launch_api_fraction\': union_us(launches)/span,\n            \'host_outside_torch_ops_fraction\': 1-union_us(ops)/span,\n            \'fractions_overlap\': True, \'idle_is_not_a_direct_measure_of_python_cost\': True,\n            \'sync_event_count\': len(syncs)}\n\n\ndef profile(model, encoded, order, variant, device, output, deadline):\n    C.check_deadline(deadline)\n    selected = order[:256]  # K4 = 1024 lanes, same as timed sampling batches.\n    src = nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in selected],\n                batch_first=True, padding_value=C.PAD).to(device)\n    context = Fast.Context(model, src, 32)\n    torch.cuda.synchronize()\n    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,\n                                            torch.profiler.ProfilerActivity.CUDA],\n                                record_shapes=False, profile_memory=False, with_stack=False) as prof:\n        with torch.profiler.record_function(\'decode_window\'):\n            if variant == \'baseline\':\n                Fast.decode(context, 4, .7)\n            else:\n                Speed.decode(context, 4, .7, options=Speed.Options(**VARIANTS[variant]))\n    path = output.with_suffix(\'.json\')\n    prof.export_chrome_trace(str(path))\n    trace = json.loads(path.read_text())\n    summary = trace_summary(trace)\n    summary[\'operators\'] = [{\'name\': e.key, \'calls\': e.count, \'self_cpu_us\': e.self_cpu_time_total,\n        \'self_device_us\': getattr(e, \'self_device_time_total\', 0.)}\n        for e in sorted(prof.key_averages(), key=lambda e: e.self_cpu_time_total, reverse=True)[:25]]\n    with gzip.open(output.with_suffix(\'.json.gz\'), \'wt\', compresslevel=6) as stream:\n        json.dump(trace, stream, separators=(\',\', \':\'))\n    path.unlink()\n    write(output.with_suffix(\'.summary.json\'), summary)\n    return summary\n\n\ndef stratified(order, count):\n    if count >= len(order):\n        return list(order)\n    return [order[(j*(len(order)-1))//(count-1)] for j in range(count)]\n\n\ndef run(args):\n    out = args.output\n    out.mkdir(parents=True, exist_ok=True)\n    report = {\'status\': \'running\', \'training\': False, \'voting\': False, \'model\': \'C-main\',\n              \'rows\': [], \'profiles\': {}, \'greedy_checks\': [], \'GPU_gate_passed\': False,\n              \'matrix_scope\': args.scope, \'ablation_questions\': args.ablation_questions,\n              \'cuda_graphs\': {\'supported\': False, \'reason\': Speed.GRAPH_REFUSAL},\n              \'preallocated_kv\': \'already in baseline; identity alias, no redundant timed run\',\n              \'torch_threads\': \'already 1 in baseline; identity alias\',\n              \'RNG\': \'seed 270042 per run; compaction/race_sampler may change stream, categorical law unchanged\',\n              \'load\': \'sustained synthetic numeric programs through real final_executor, not measured candidate mix\',\n              \'timer\': \'generation only incl. input transfer/context/encode/output D2H; excludes load start/stop, voting, restore\',\n              \'session_start\': args.start}\n    def save():\n        report[\'session_seconds\'] = time.monotonic()-args.start\n        full_checks = [r for r in report[\'greedy_checks\'] if r[\'rows\'] == 1919 and r[\'variant\'] == \'all\']\n        report[\'full_greedy_bit_exact\'] = (not any(r[\'mismatch_rows\'] for r in full_checks)\n                                          if len(full_checks) == 2 else None)\n        comparisons = []\n        for row in report[\'rows\']:\n            if row[\'variant\'] == \'all\' and row[\'scope\'] == \'test1919\':\n                control = next(b for b in report[\'rows\'] if b[\'variant\'] == \'baseline\' and\n                    b[\'scope\'] == row[\'scope\'] and b[\'mode\'] == row[\'mode\'] and\n                    bool(b[\'load_workers\']) == bool(row[\'load_workers\']))\n                comparisons.append({\'mode\': row[\'mode\'], \'loaded\': bool(row[\'load_workers\']),\n                                    \'speedup\': control[\'seconds\']/row[\'seconds\']})\n        report[\'full_speedups\'] = comparisons\n        report[\'target_2x_all_generation\'] = (all(r[\'speedup\'] >= 2 for r in comparisons)\n                                            if len(comparisons) == 4 else None)\n        write(out/\'summary.json\', report)\n        if report[\'rows\']:\n            fields = [\'variant\', \'mode\', \'scope\', \'load_workers\', \'nice\', \'questions\', \'sequences\',\n                \'nice_requested_increment\', \'nice_applied\',\n                \'seconds\', \'sequences_per_second\', \'parent_cpu_seconds\', \'gpu_stream_span_seconds\',\n                \'peak_allocated_bytes\', \'peak_reserved_bytes\', \'greedy_mismatches\']\n            with (out/\'timings.csv\').open(\'w\', newline=\'\') as stream:\n                writer = csv.DictWriter(stream, fieldnames=fields, extrasaction=\'ignore\')\n                writer.writeheader()\n                writer.writerows(report[\'rows\'])\n    save()\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.use_deterministic_algorithms(True)\n    torch.utils.deterministic.fill_uninitialized_memory = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    device = torch.device(\'cuda:0\')\n    assert os.environ.get(\'CUDA_VISIBLE_DEVICES\') == \'0\'\n    assert torch.cuda.device_count() == 1 and \'T4\' in torch.cuda.get_device_name(0)\n    assert str(torch.__version__) == \'2.11.0+cu128\' and torch.version.cuda == \'12.8\'\n    report[\'runtime\'] = {\'torch\': str(torch.__version__), \'cuda\': torch.version.cuda,\n                         \'device\': torch.cuda.get_device_name(0), \'cpu_count\': os.cpu_count(),\n                         \'affinity\': sorted(os.sched_getaffinity(0)), \'torch_threads\': torch.get_num_threads()}\n    inputs_dir = out/\'inputs\'\n    inputs_dir.mkdir(exist_ok=True)\n    inputs = prepare(args.inputs, inputs_dir)\n    rows = json.loads(inputs[\'test.json\'].read_text())\n    assert len(rows) == 1919\n    tok, _ = tokenizer(inputs[\'tokenizer.json\'], 30)\n    encoded = [[C.BOS]+C.encode(tok, r[\'question\'])+[C.EOS] for r in rows]\n    order = sorted(range(1919), key=lambda i: (len(encoded[i]), i))\n    subset = stratified(order, args.ablation_questions)\n    write(out/\'row-ids.json\', {\'full_sorted\': order, \'ablation_length_stratified\': subset})\n    model = load_model(unique(args.live, \'model-C-main__weights__l2-62280-fp16.pt\'), device)\n    # Warm cuBLAS/SDPA for both initial batch shapes, no reported cold-start speedup.\n    for k in (1, 4):\n        measure(model, encoded, order[:1024//k], \'baseline\', k, device, args.deadline, cap=8)\n    report[\'setup_seconds\'] = time.monotonic()-args.start\n    save()\n    references = {}\n\n    def one(variant, k, load, selected, scope):\n        load.alive()\n        row, predictions = measure(model, encoded, selected, variant, k, device, args.deadline)\n        row.update(scope=scope, load_workers=load.workers, nice=load.info[\'nice\'],\n                   nice_requested_increment=load.nice, nice_applied=load.info[\'nice_applied\'])\n        if k == 1:\n            # Subset batches have different GEMM shapes from full1919: use their\n            # own paired control, never misattribute shape rounding to a flag.\n            reference = references.setdefault(scope, {})\n            missing = [i for i in selected if i not in reference]\n            if missing:\n                assert variant == \'baseline\', \'Control must precede each greedy comparison\'\n                reference.update(predictions)\n            mismatches = [i for i in selected if not torch.equal(predictions[i], reference[i])]\n            row[\'greedy_mismatches\'] = len(mismatches)\n            report[\'greedy_checks\'].append({\'variant\': variant, \'load_workers\': load.workers,\n                \'nice\': load.nice, \'rows\': len(selected), \'mismatch_rows\': mismatches})\n        else:\n            row[\'greedy_mismatches\'] = None\n        report[\'rows\'].append(row)\n        load.alive()\n        save()\n        print(json.dumps({key: row[key] for key in [\'variant\', \'mode\', \'scope\', \'load_workers\',\n                          \'seconds\', \'sequences_per_second\', \'greedy_mismatches\']}), flush=True)\n\n    try:\n        # Mandatory full-data control/all gate first, both with and without sustained execution.\n        for loaded in (False, True):\n            for variant in (\'baseline\', \'all\'):\n                workers = (2 if variant == \'all\' else 4) if loaded else 0\n                nice = 10 if loaded and variant == \'all\' else 0\n                with Load(out/f\'load-full-{variant}-{workers}\', workers, nice) as load:\n                    for k in (1, 4):\n                        one(variant, k, load, order, \'test1919\')\n                    report[\'profiles\'][f\'{variant}-workers{workers}\'] = profile(\n                        model, encoded, order, variant, device, out/f\'trace-{variant}-workers{workers}\', args.deadline)\n                    save()\n        # Reuse mandatory rows for baseline/all; single flags share a paired ablation control.\n        selected = order if args.scope == \'full\' else subset\n        for loaded in (False, True):\n            for variant in VARIANTS:\n                if args.scope == \'full\' and variant in (\'baseline\', \'all\'):\n                    continue\n                # workers/nice have no effect with execution off: explicit aliases below.\n                if not loaded and variant in (\'workers2\', \'nice10\'):\n                    continue\n                workers = (2 if variant in (\'workers2\', \'all\') else 4) if loaded else 0\n                nice = 10 if loaded and variant in (\'nice10\', \'all\') else 0\n                with Load(out/f\'load-ablation-{variant}-{workers}\', workers, nice) as load:\n                    for k in (1, 4):\n                        one(variant, k, load, selected, \'test1919\' if args.scope == \'full\' else \'test1919-subset\')\n        report[\'identity_aliases\'] = [\n            {\'variant\': \'preallocated_kv\', \'same_as\': \'baseline\', \'reason\': \'self-KV index writes and one-time cross projection already present\'},\n            {\'variant\': \'torch_threads1\', \'same_as\': \'baseline\', \'reason\': \'v5 already has one intra/inter-op thread\'},\n            {\'variant\': \'workers2/nice10 without load\', \'same_as\': \'baseline\', \'reason\': \'no executor exists\'}]\n        report[\'matrix_complete\'] = True\n        full_checks = [r for r in report[\'greedy_checks\'] if r[\'rows\'] == 1919 and r[\'variant\'] == \'all\']\n        report[\'GPU_gate_passed\'] = (len(full_checks) == 2 and\n                                    not any(r[\'mismatch_rows\'] for r in report[\'greedy_checks\']))\n        report[\'status\'] = \'complete\' if report[\'GPU_gate_passed\'] else \'greedy_mismatch\'\n        report[\'nice_supported\'] = all(r[\'nice_applied\'] for r in report[\'rows\']\n                                       if r[\'nice_requested_increment\'])\n    except C.BudgetExpired:\n        report.update(status=\'budget_stop_partial\', matrix_complete=False, GPU_gate_passed=False)\n    except BaseException as error:\n        report.update(status=\'error\', error=repr(error), matrix_complete=False, GPU_gate_passed=False)\n        raise\n    finally:\n        save()\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--inputs\', type=Path, required=True)\n    parser.add_argument(\'--live\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--start\', type=float, required=True)\n    parser.add_argument(\'--deadline\', type=float, required=True)\n    parser.add_argument(\'--scope\', choices=[\'full\', \'paired-subset\'], default=\'paired-subset\')\n    parser.add_argument(\'--ablation-questions\', type=int, default=192)\n    args = parser.parse_args()\n    assert 2 <= args.ablation_questions <= 1919\n    try:\n        run(args)\n    except BaseException as error:\n        # Cover setup/runtime assertions too, before the main measurement try block.\n        path = args.output/\'summary.json\'\n        report = json.loads(path.read_text()) if path.exists() else {}\n        report.update(status=\'error\', error=repr(error), GPU_gate_passed=False,\n                      matrix_complete=False, session_seconds=time.monotonic()-args.start)\n        write(path, report)\n        raise\n', 'final_profile_load.py': '"""Torch-free bounded workload driver, using the real frozen final executor."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport sys\nimport time\nfrom final_executor import Executor\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--workers\', type=int, required=True)\n    parser.add_argument(\'--nice\', type=int, default=0)\n    parser.add_argument(\'--directory\', type=Path, required=True)\n    args = parser.parse_args()\n    # Service and spawn workers inherit priority; GPU Python priority is untouched.\n    nice_error = None\n    try:\n        effective_nice = os.nice(args.nice) if args.nice else os.getpriority(os.PRIO_PROCESS, 0)\n    except PermissionError as error:\n        # macOS sandbox can deny even LOWERING priority. Record, never bypass it.\n        effective_nice, nice_error = os.getpriority(os.PRIO_PROCESS, 0), str(error)\n    stop = False\n    def stopping(*_):\n        nonlocal stop\n        stop = True\n    signal.signal(signal.SIGTERM, stopping)\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            return\n    executor = Executor(args.workers)\n    started = time.monotonic()\n    batches, completed = 0, 0\n    try:\n        executor.submit([{\'code\': \'def solve():\\n    return 1\\n\'}])\n        assert executor.drain(time.monotonic()+20.), \'Load worker startup failed\'\n        (args.directory/\'ready.json\').write_text(json.dumps({\'workers\': args.workers,\n            \'nice\': effective_nice, \'nice_requested_increment\': args.nice,\n            \'nice_applied\': nice_error is None, \'nice_error\': nice_error}))\n        while not stop and time.monotonic()-started < 660:\n            # Same jobs and rate across variants. Numeric AST/execution path, no training/gold.\n            # ~0.02 CPU sec/task on T4 host, sustained rather than bursty pipeline load.\n            jobs = [{\'code\': f\'def solve():\\n    x = {batches*32+i}\\n    for j in range(250000):\\n        x = (x+j)%1000003\\n    return x\\n\'}\n                    for i in range(32)]\n            executor.submit(jobs)\n            while not stop and not executor.drain(time.monotonic()+.05):\n                pass\n            batches += 1\n            completed = len(executor.results)\n    finally:\n        report = executor.summary()\n        report.update(batches=batches, completed=completed, nice=effective_nice, nice_error=nice_error,\n                      workload=\'synthetic sustained numeric programs through frozen executor\')\n        (args.directory/\'load-summary.json\').write_text(json.dumps(report, indent=2))\n        executor.close()\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name, source in SOURCES.items():
    (MODULES/name).write_text(source)
stop_watchdog = threading.Event()
children = set()
def kill_children():
    for pid in list(children):
        try:
            os.killpg(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
def watch():
    while not stop_watchdog.wait(.2):
        if time.monotonic()-SESSION_START >= 690:
            kill_children()
            (OUTPUT/'watchdog.json').write_text(json.dumps({'triggered': True, 'seconds': time.monotonic()-SESSION_START}))
            return
threading.Thread(target=watch, daemon=True, name='decode-profile-watchdog').start()
print('Private offline profile; no API calls, no training; watchdog690s')


In [ ]:
SCOPE = 'paired-subset'
def find_input(slug, root=Path('/kaggle/input')):
    direct = root/slug
    if direct.is_dir():
        return direct
    hits = [p for p in root.rglob(slug) if p.is_dir()]
    assert len(hits) == 1, f'Cannot resolve mount {slug}: {hits}'
    return hits[0]
inputs = find_input('nlp-hw02-26b-inputs')
live = find_input('nlp-hw02-l3-live')
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', CUBLAS_WORKSPACE_CONFIG=':4096:8',
           OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', PYTHONHASHSEED='0')
command = [sys.executable, str(MODULES/'final_profile.py'), '--inputs', str(inputs), '--live', str(live),
           '--output', str(OUTPUT), '--start', str(SESSION_START), '--deadline', str(SESSION_START+650),
           '--scope', SCOPE, '--ablation-questions', '192']
try:
    with (OUTPUT/'run.log').open('w') as log:
        child = subprocess.Popen(command, env=env, start_new_session=True, stdout=log, stderr=subprocess.STDOUT)
        children.add(child.pid)
        while child.poll() is None:
            if time.monotonic()-SESSION_START >= 695:
                kill_children()
                break
            time.sleep(.2)
        child.wait(timeout=3)
        children.discard(child.pid)
    report = json.loads((OUTPUT/'summary.json').read_text())
    print(json.dumps({k: report.get(k) for k in ['status', 'matrix_complete', 'GPU_gate_passed', 'full_greedy_bit_exact',
                     'full_speedups', 'target_2x_all_generation', 'session_seconds']}, indent=2))
    if (OUTPUT/'timings.csv').exists():
        print((OUTPUT/'timings.csv').read_text())
    print('Exit:', child.returncode, 'artifacts: /kaggle/working/decode-profile')
finally:
    kill_children()
    stop_watchdog.set()


Скачать decode-profile/summary.json, timings.csv, trace-*.summary.json и trace-*.json.gz. GPU_gate_passed означает отсутствие greedy token mismatches; target_2x_all_generation проверяется отдельно. Проверка CPU и готовность бандла не заменяют эти GPU gates. Если матрица partial — не делать выводов по отсутствующим ячейкам.